# Wavefront reconstruction from lantern signals at one wavelength

Reconstructs the first 5 DM modes (the petal pistons of the bundled ANDES
modal basis) from the output-core signals of the photonic lantern at its
design wavelength, with an MLP and with explainable alternatives: a linear
reconstructor (optionally with symbolic-regression corrections), and sparse
or dense polynomial fits on the raw core signals or on their Zernike
projection.

The coefficients are uniform in +-span (`ml_reconstruct.presets`: +-lambda/2,
400 nm for pl19_800, 600 nm at 1200 nm), the range where a single piston is
unambiguous at one wavelength (see `trainMLP_multiwavelength.ipynb`
/ `PL_EML.ipynb` for larger pistons). Two petals can still differ by twice
that, and how well a lantern resolves this depends on its port count: with 7
ports (6 independent normalised signals for 5 pistons) many piston sets give
nearly the same signals over the range, with 19 ports far fewer do.
At exactly +-lambda/2 a piston equals its opposite, so the edges of the range
are ambiguous for every lantern.
Fitted models are cached under the lantern's data folder; set `RETRAIN = True`
to refit.

The lantern is the preset named by the environment variable `CBEAM_LANTERN`
(`pl19_800`, the reference 19-port lantern at 800 nm, by default; or
`pl7_1200`, `pl19_1200` -- see `batch_pipeline.LANTERNS`). Executed copies
for each lantern are in `runs/<lantern>/`.


In [ ]:
import os
os.environ["CBEAM_BACKEND"] = "numpy"

import numpy as np
import matplotlib.pyplot as plt
import ml_reconstruct as ml

from ml_reconstruct.presets import settings
S = settings()                  # this lantern's settings (CBEAM_LANTERN)
LANTERN, DATA = S["lantern"], S["data_dir"]
N_SAMPLES, N_TRAIN = 10000, 8000
M_MODES = 5                     # DM modes 0..4 (petal pistons)
AMPLITUDE_NM = S["single_span_nm"]
DATASET = f"{DATA}/single.npz" if LANTERN != "pl19_800" else "data/ml/single_800nm.npz"
cache = ml.ModelCache(f"{DATA}/cache_single", retrain=False)
print(f"lantern {LANTERN}, pistons +-{AMPLITUDE_NM:.0f} nm, data in {DATA}")

## Dataset

Each sample is a random configuration of the 5 modes, propagated through
the lantern; the signal of each core is the mean of a 5x5-pixel window of
the imaged output face around the core centre, with the input normalised to
unit coupled power (see `ml_reconstruct.simulate`). Simulating 10000 samples
takes 15-40 minutes on the numpy backend.

In [ ]:
if os.path.exists(DATASET):
    ds = ml.LanternDataset.load(DATASET)
else:
    import batch_pipeline as bp
    from ml_reconstruct.simulate import simulate_core_images
    p = bp.get_simulation_parameters()
    ifunc = bp.load_ifunc(p["ifunc_file"]); bp.center_ifunc_mask(ifunc)
    coeff, _ = bp.create_random_aberration_configs(
        N_SAMPLES, M_MODES, -AMPLITUDE_NM, AMPLITUDE_NM, rng=0)
    ds = simulate_core_images(p, ifunc, coeff, N_TRAIN)
    os.makedirs(os.path.dirname(DATASET), exist_ok=True)
    ds.save(DATASET)

S_tr, C_tr = ds.train
S_va, C_va = ds.val
xs, ys = ml.Standardizer().fit(S_tr), ml.Standardizer().fit(C_tr)   # training split only
Xtr, Xva, Ytr = xs.transform(S_tr), xs.transform(S_va), ys.transform(C_tr)
print(f"{len(S_tr)} training / {len(S_va)} validation samples, {S_tr.shape[1]} cores, "
      f"{C_tr.shape[1]} modes, coefficients std {C_tr.std():.0f} nm")

RESULTS = {}                    # model -> per-mode validation RMSE [nm]
def record(name, Y_pred_std):
    """Score standardised predictions against the validation coefficients."""
    RESULTS[name] = ml.rmse_nm(ys.inverse(Y_pred_std), C_va)
    print(f"{name:16s} mean {RESULTS[name].mean():6.1f} nm   per mode {np.round(RESULTS[name], 1)}")

## MLP

Three hidden layers of 128 units, Adam, 500 epochs; the weights of the epoch
with the lowest validation loss are kept.

In [ ]:
LAYERS = [Xtr.shape[1], 128, 128, 128, M_MODES]
MLP_FILE = f"{DATA}/cache_single/mlp-{'x'.join(map(str, LAYERS))}-e500.npz"
if os.path.exists(MLP_FILE) and not cache.retrain:
    params, _, extra = ml.load_mlp(MLP_FILE)
    hist = {"train": extra["train"], "val": extra["val"], "best_epoch": int(extra["best_epoch"])}
else:
    params, hist = ml.train_mlp(LAYERS, Xtr, Ytr, Xva, ys.transform(C_va),
                                epochs=500, learning_rate=3e-4)
    os.makedirs(os.path.dirname(MLP_FILE), exist_ok=True)
    ml.save_mlp(MLP_FILE, params, LAYERS, train=hist["train"], val=hist["val"],
                best_epoch=hist["best_epoch"])
print(f"best epoch {hist['best_epoch']}")
record("MLP", np.asarray(ml.mlp_forward(params, Xva)))

C_mlp = ys.inverse(np.asarray(ml.mlp_forward(params, Xva)))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist["train"], label="train"); ax[0].plot(hist["val"], label="validation")
ax[0].set_yscale("log"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("Huber loss"); ax[0].legend()
k = 2
ax[1].scatter(C_va[:, k], C_mlp[:, k], s=4, alpha=0.4)
ax[1].plot([-AMPLITUDE_NM, AMPLITUDE_NM], [-AMPLITUDE_NM, AMPLITUDE_NM], "r--")
ax[1].set_xlabel("true coefficient (nm)"); ax[1].set_ylabel("MLP estimate (nm)")
ax[1].set_title(f"mode {k}")
plt.tight_layout(); plt.show()

## Baselines: linear reconstructor and k-nearest neighbours

The linear reconstructor is a 5 x n_cores matrix (ridge-regularised); k-NN needs no
training and shows how well the signals determine the coefficients at all.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor

linear = cache("linear", lambda: Ridge(1.0).fit(Xtr, Ytr))
record("linear", linear.predict(Xva))
knn = cache("kNN", lambda: KNeighborsRegressor(5, weights="distance").fit(Xtr, Ytr))
record("kNN", knn.predict(Xva))

## Optional: linear reconstructor + symbolic-regression residual

PySR searches, per mode, for a compact formula of the core signals that
explains what the linear reconstructor misses. It needs PySR (and Julia),
and only improves the linear model marginally, so it is off by default.

In [ ]:
RUN_SR = False

def _inv(x):                    # module-level, so the fitted model can be pickled
    return 1 / x

def fit_linear_sr():
    from pysr import PySRRegressor
    residual = Ytr - linear.predict(Xtr)
    models = []
    for k in range(M_MODES):
        m = PySRRegressor(
            niterations=100, populations=41, population_size=60, maxsize=35,
            binary_operators=["+", "-", "*"], unary_operators=["square", "inv(x) = 1/x", "sin"],
            elementwise_loss="L2DistLoss()", parsimony=0.0, model_selection="best",
            parallelism="multithreading", turbo=True, random_state=0, verbosity=0,
            progress=False, extra_sympy_mappings={"inv": _inv})
        models.append(m.fit(Xtr, residual[:, k], variable_names=[f"f{i}" for i in range(Xtr.shape[1])]))
    return models

if RUN_SR:
    sr = cache("linear+SR", fit_linear_sr)
    record("linear+SR", linear.predict(Xva) + np.stack([m.predict(Xva) for m in sr], axis=1))
    for k, m in enumerate(sr):
        print(f"  mode {k}: residual = {m.get_best()['equation']}")

## Sparse degree-3 polynomial (ElasticNet)

All monomials of degree <= 3 in the core signals (119 for 7 cores, 1539 for
19), standardised, with
an ElasticNet whose L1 part zeroes most of them (penalty chosen by 5-fold
cross-validation).

In [ ]:
from sklearn.linear_model import ElasticNetCV

Z3_tr, Z3_va, _, _ = ml.poly_features(Xtr, Xva, 3)
elastic = cache("elastic-d3", lambda: [
    ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.8, 1.0], max_iter=100000, cv=5,
                 n_jobs=-1, random_state=0).fit(Z3_tr, Ytr[:, k]) for k in range(M_MODES)])
print("non-zero terms per mode:", [int(np.count_nonzero(m.coef_)) for m in elastic])
record("elastic-d3", np.stack([m.predict(Z3_va) for m in elastic], axis=1))

## Zernike representation of the core signals

The core signals, placed at their cores on the unit disk, projected onto the
first `n_zern` Noll Zernikes (36 for 19 cores; 10 for 7, the fewest with full
rank on one hexagonal ring, where coma coincides with tilt). The sampled basis
has rank n_cores, so this is a change
of variables rather than a compression: the degree-3 polynomials of the
Zernike coefficients span the same functions as those of the raw signals,
and the two representations differ only in how the fits below regularise
and select terms.

In [ ]:
N_ZERN = S["n_zern"]
B, znames = ml.zernike_basis(ds.core_xy, N_ZERN, fill=0.8)
P = ml.modal_projector(B)
print(f"Zernike basis {B.shape}, rank {np.linalg.matrix_rank(B)}")

zs = ml.Standardizer(eps=1e-9).fit(S_tr @ P.T)
Ztr_modal, Zva_modal = zs.transform(S_tr @ P.T), zs.transform(S_va @ P.T)

s_mean = S_tr.mean(axis=0)
xy = ds.core_xy / np.linalg.norm(ds.core_xy, axis=1).max()
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for a, s, title in [(ax[0], s_mean, "mean core signal"), (ax[1], B @ (P @ s_mean), "Zernike reconstruction")]:
    sc = a.scatter(xy[:, 0], xy[:, 1], c=s, s=350)
    for i, (x, y) in enumerate(xy):
        a.text(x, y, str(i), ha="center", va="center", fontsize=7, color="w")
    a.set_aspect("equal"); a.set_title(title); plt.colorbar(sc, ax=a)
plt.tight_layout(); plt.show()

## Dense degree-3 polynomial ridge: Zernike vs. raw core signals

In [ ]:
for tag, F_tr, F_va in [("zern-d3", Ztr_modal, Zva_modal), ("fiber-d3", Xtr, Xva)]:
    model = cache(tag, lambda F_tr=F_tr: ml.fit_dense_poly_ridge(F_tr, Ytr, 3))
    print(f"{model[0].n_output_features_:5d} terms   ", end="")
    record(tag, ml.predict_dense_poly_ridge(model, F_va))

## Compact model: OMP with a term budget K

Orthogonal Matching Pursuit keeps at most K of the degree-3 monomials of the
Zernike coefficients per mode (they span the same functions as the raw-signal
cubic) (after screening them by correlation with
the mode). The sweep shows the accuracy / size trade-off.

In [ ]:
SCREEN = 9000
KS = S["omp_ks"]
RECORD_KS = S["omp_record"]

Zd3_tr, Zd3_va, _, poly_d3 = ml.poly_features(Ztr_modal, Zva_modal, 3)
def omp_fits(K):
    return cache(f"zern-omp-K{K}", lambda: [ml.omp_fit(Zd3_tr, Ytr[:, k], K, SCREEN)
                                            for k in range(M_MODES)])
def omp_predict(fits):
    return np.stack([ml.omp_predict(Zd3_va, f) for f in fits], axis=1)

sweep = {}
for K in KS:
    fits = omp_fits(K)
    sweep[K] = (np.mean([ml.n_terms(f) for f in fits]),
                ml.rmse_nm(ys.inverse(omp_predict(fits)), C_va).mean())
    print(f"K={K:5d}: {sweep[K][0]:7.1f} terms/mode   mean RMSE {sweep[K][1]:5.1f} nm")
for K in RECORD_KS:
    record(f"zern-omp-K{K}", omp_predict(omp_fits(K)))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([v[0] for v in sweep.values()], [v[1] for v in sweep.values()], "ks-")
ax.set_xlabel("terms per mode"); ax.set_ylabel("mean validation RMSE (nm)")
ax.set_title("Zernike degree-3 OMP: accuracy vs. model size"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Comparison

In [ ]:
order = ["linear", "linear+SR", "elastic-d3", "fiber-d3", "zern-d3",
         *[f"zern-omp-K{K}" for K in RECORD_KS], "MLP", "kNN"]
names = [m for m in order if m in RESULTS]
for m in names:
    print(f"  {m:16s} {RESULTS[m].mean():6.1f} nm   per mode {np.round(RESULTS[m], 1)}")

x, w = np.arange(M_MODES), 0.8 / len(names)
plt.figure(figsize=(13, 5))
for i, m in enumerate(names):
    plt.bar(x + (i - len(names) / 2 + 0.5) * w, RESULTS[m], w, label=m)
plt.axhline(RESULTS["kNN"].mean(), color="k", ls="--", lw=1)
plt.yscale("log"); plt.xticks(x); plt.xlabel("DM mode"); plt.ylabel("validation RMSE (nm)")
plt.title(f"Reconstructors, {LANTERN} (dashed: k-NN mean)"); plt.legend(ncol=3, fontsize=8)
plt.grid(True, axis="y", ls=":", alpha=0.5); plt.tight_layout(); plt.show()

## Which monomials does OMP keep?

Kept vs. available monomials per degree at K = `k_table`, and how the degree mix
of the kept terms changes with the budget.

In [ ]:
K_TABLE, KS_MIX = S["k_table"], S["ks_mix"]
degree = poly_d3.powers_.sum(axis=1)
degs = np.arange(1, degree.max() + 1)
total = np.array([(degree == d).sum() for d in degs])
def kept_by_degree(K):
    kept = [f[1][np.flatnonzero(f[0].coef_)] for f in omp_fits(K)]
    return np.array([[np.sum(degree[g] == d) for d in degs] for g in kept])

kept = kept_by_degree(K_TABLE)
print(f"K = {K_TABLE}: kept / available monomials per degree (one column per mode)")
for i, d in enumerate(degs):
    print(f"  degree {d}: {total[i]:5d} available, kept {kept[:, i]}  ({kept[:, i].mean() / total[i]:.1%})")

mix = np.array([kept_by_degree(K).sum(axis=0) for K in KS_MIX], float)
mix = (mix / mix.sum(axis=1, keepdims=True)).T
plt.figure(figsize=(6, 4))
plt.stackplot(KS_MIX, *mix, labels=[f"degree {d}" for d in degs], alpha=0.85)
plt.xlabel("K (terms per mode)"); plt.ylabel("share of kept terms")
plt.title("degree mix of the kept terms"); plt.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

## Which Zernike coefficients matter?

For the largest recorded K: the total |coefficient| (in standardised units) of
the monomials each Zernike coefficient appears in, per DM mode.

In [ ]:
K = max(RECORD_KS)
contains = (poly_d3.powers_ > 0).astype(float)          # monomial i involves variable v
mass = np.zeros((M_MODES, len(znames)))
for k, (omp, sel) in enumerate(omp_fits(K)):
    nz = np.flatnonzero(omp.coef_)
    mass[k] = contains[sel[nz]].T @ np.abs(omp.coef_[nz])

top = np.argsort(-mass.sum(axis=0))[:len(znames)]
plt.figure(figsize=(10, 5))
bottom = np.zeros(len(top))
for k in range(M_MODES):
    plt.bar([znames[i] for i in top], mass[k, top], bottom=bottom, label=f"mode {k}")
    bottom += mass[k, top]
plt.xticks(rotation=60); plt.ylabel("total |coefficient|")
plt.title(f"Zernike-coefficient importance in zern-omp-K{K}"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()